# Random Forest

**Chủ phần:** Đức Anh — M04 / T15, T19, T23, T27.

**Trạng thái:** Đã hoàn thiện đầy đủ 6 nội dung M04 (T15, T19, T23, T27).

Baseline RF, tuning `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`.

Dùng split/pipeline chung từ M03, `random_state=42`, 5-fold `StratifiedKFold` trên tập train và metric chính **`average_precision` (AP)**. Không dùng test để tuning/chọn mô hình.

## Nội dung cần triển khai

1. Phương pháp và giả định.
2. Dataset/hash, cấu hình và preprocessing chung.
3. Baseline và CV.
4. Hyperparameter search trên train/CV.
5. Kết quả mean/std, best_params, thời gian và fitted pipeline.
6. Nhận xét và bàn giao.

## 1. Phương pháp và giả định (T15)

### 1.1. Decision Tree (Cây quyết định)
- Decision Tree phân hoạch đệ quy không gian đặc trưng bằng cách chọn đặc trưng $j$ và ngưỡng tách $t$ sao cho độ hỗn tạp (impurity) ở hai nút con giảm nhiều nhất. Hai tiêu chuẩn phổ biến trong phân loại nhị phân là:
  - **Gini impurity** (mặc định trong `scikit-learn`):
    $$G(m) = 1 - \sum_{k \in \{0, 1\}} p_{m,k}^2 = 2 p_{m,1}(1 - p_{m,1})$$
  - **Entropy (Information Gain):**
    $$H(m) = -\sum_{k \in \{0, 1\}} p_{m,k} \log_2 p_{m,k}$$
- **Ưu điểm:** Học được quan hệ phi tuyến và tương tác giữa các đặc trưng; bất biến với phép biến đổi đơn điệu nên **không yêu cầu chuẩn hóa thang đo (`StandardScaler`)**.
- **Hạn chế:** Cây không giới hạn độ sâu dễ ghi nhớ nhiễu (**overfitting**), có **phương sai (variance) cao** — chỉ cần thay đổi nhỏ trong tập huấn luyện, cấu trúc cây có thể thay đổi mạnh.

### 1.2. Bagging (Bootstrap Aggregating)
- Tạo $B$ tập huấn luyện con bằng cách lấy mẫu có hoàn lại (**bootstrap sampling**) với cùng kích thước $N$ từ tập train, huấn luyện độc lập $B$ cây quyết định, rồi tổng hợp xác suất dự đoán:
  $$\hat{p}(y = 1 \mid x) = \frac{1}{B} \sum_{b=1}^{B} \hat{p}_b(y = 1 \mid x)$$
- **Tác dụng:** Giảm mạnh **variance** mà gần như không làm tăng **bias**. Tuy nhiên, nếu dữ liệu có một số đặc trưng rất mạnh (như `GenHlth`, `HighBP`, `BMI`, `Age`), hầu hết các cây trong Bagging đều chọn cùng đặc trưng ở các nút trên cùng, khiến các cây **tương quan cao** với nhau và giới hạn mức giảm variance.

### 1.3. Random Forest
- Mở rộng Bagging bằng cơ chế **chọn ngẫu nhiên một tập con đặc trưng** kích thước `max_features` tại **mỗi nút chia** (thay vì xét toàn bộ $p = 21$ đặc trưng).
- Cơ chế này làm giảm tương quan giữa các cây (decorrelation), giúp phép lấy trung bình xác suất giảm variance hiệu quả hơn nữa.
- **Các giả định và lưu ý với bộ dữ liệu CDC Diabetes (21 đặc trưng):**
  1. Dữ liệu gồm 14 biến nhị phân (`0/1`), 4 biến thứ bậc (`GenHlth`, `Age`, `Education`, `Income`) và 3 biến đếm/liên tục nguyên (`BMI`, `MentHlth`, `PhysHlth`) — phù hợp với mô hình cây và **không cần scaling**.
  2. Nhãn mất cân bằng (`~15.29%` lớp 1): với ngưỡng mặc định `0.5`, cây có xu hướng thiên về lớp đa số (`0`), làm `Accuracy` cao nhưng `Recall(1)` thấp. Do đó, đánh giá bằng **`Average Precision` (AP)** trên xác suất liên tục `predict_proba[:, 1]` phản ánh đúng chất lượng xếp hạng nguy cơ.
  3. `feature_importances_` (Mean Decrease in Impurity — MDI) hữu ích để diễn giải, nhưng có thể hơi thiên vị các biến có nhiều giá trị rời rạc hơn (như `BMI`, `Age`) so với biến nhị phân.

### 1.4. Vai trò 4 siêu tham số cần tối ưu (T23)
| Tham số | Ý nghĩa | Ảnh hưởng đến Bias / Variance & AP |
|---|---|---|
| `n_estimators` | Số lượng cây trong rừng | Càng nhiều cây thì xác suất `predict_proba` càng ổn định (giảm variance), nhưng thời gian train và kích thước file `.joblib` tăng tuyến tính. |
| `max_depth` | Độ sâu tối đa của mỗi cây | Giới hạn độ sâu giúp chặn overfitting trên tập lớn (~183k mẫu), tránh các nút lá quá đặc thù. |
| `min_samples_leaf` | Số mẫu tối thiểu ở mỗi nút lá | Tăng `min_samples_leaf` giúp ước lượng xác suất ở nút lá mượt hơn (ít bị 0.0 hoặc 1.0 cực đoan), thường cải thiện rõ rệt **Average Precision** và **ROC-AUC**. |
| `max_features` | Số đặc trưng xét tại mỗi nút chia | Nhỏ hơn (`sqrt`, `log2`) giúp giảm tương quan giữa các cây; lớn hơn (ví dụ `0.5`) giúp từng cây mạnh hơn (giảm bias). |

## 2. Dataset/hash, cấu hình và preprocessing chung

Nạp cấu hình từ `configs/experiment.yaml`, kiểm tra `data/splits/split_metadata.json`, đọc tập train 80% qua API chung `src.data_split.load_train_data()` (không mở tập test), và kiểm tra cấu trúc pipeline từ `src.preprocessing.build_pipeline(..., model_type="random_forest")`.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().resolve()
if not (ROOT / "configs" / "experiment.yaml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_split import load_config, load_train_data
from src.preprocessing import build_pipeline, get_cv_splitter

config = load_config()
split_meta_path = ROOT / config["paths"]["splits"] / "split_metadata.json"
split_meta = json.loads(split_meta_path.read_text(encoding="utf-8"))

X_train, y_train = load_train_data(config=config)
cv = get_cv_splitter(config)

print(f"Dataset version     : {config['dataset']['version']}")
print(f"Raw SHA-256 (config): {config['dataset']['sha256']}")
print(f"Clean CSV SHA-256   : {split_meta['source_file_sha256']}")
print(f"Train indices hash  : {split_meta['train_indices_sha256']}")
print(f"X_train shape       : {X_train.shape}")
print(f"Positive rate (y=1) : {y_train.mean():.4%} ({int((y_train == 1).sum()):,} / {len(y_train):,})")
print(f"Missing cells       : {int(X_train.isna().sum().sum())}")
print(f"CV splitter         : {cv}")
print(f"Primary metric      : {config['training']['primary_metric']}")

Dataset version     : uci891_binary_imbalanced
Raw SHA-256 (config): 9f71fda9d4ae5f4878c99b9233b6a16accfa9a17c194116a6b78100540934964
Clean CSV SHA-256   : 560b9d35f968157093c8eeb52da4e1f27a181881cbb6207f15f6999f123071e0
Train indices hash  : 87d3245e2297bd35727cdf557d0fdc12da069ca6d39ce8394c45d0f827be5c85
X_train shape       : (183579, 21)
Positive rate (y=1) : 15.2948% (28,078 / 183,579)
Missing cells       : 0
CV splitter         : StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
Primary metric      : average_precision


In [2]:
# Kiểm tra pipeline chung cho Random Forest (chưa fit — chỉ kiểm tra cấu trúc)
rf_demo_pipeline = build_pipeline(
    RandomForestClassifier(random_state=config["split"]["random_state"], n_jobs=-1),
    X_train,
    model_type="random_forest",
)
rf_demo_pipeline

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


## 3. Baseline và CV (T19)

Huấn luyện mô hình **Random Forest mặc định** (`RandomForestClassifier(random_state=42, n_jobs=-1)`) bọc trong pipeline chung của M03, đánh giá bằng `StratifiedKFold(5)` trên tập train (`X_train, y_train`) qua `src.models.random_forest.run_baseline()`.

In [3]:
from src.models.random_forest import run_baseline

baseline_result = run_baseline(X_train, y_train, config=config)
baseline_cv = baseline_result["validation_cv"]

metrics_order = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
baseline_table = pd.DataFrame(
    {
        "Metric": metrics_order,
        "Mean": [baseline_cv[f"{m}_mean"] for m in metrics_order],
        "Std": [baseline_cv[f"{m}_std"] for m in metrics_order],
    }
)
print(f"Thời gian chạy 5-fold CV (Baseline): {baseline_result['training_time_seconds']:.2f}s")
baseline_table

Thời gian chạy 5-fold CV (Baseline): 17.36s


,Metric,Mean,Std
0,accuracy,0.844296,0.001843
1,precision,0.474119,0.016619
2,recall,0.162903,0.003199
3,f1,0.242467,0.005495
4,roc_auc,0.778680,0.000931
5,average_precision,0.362876,0.004303


## 4. Hyperparameter search trên train/CV (T23)

Tối ưu 4 siêu tham số bắt buộc (`n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`) bằng `RandomizedSearchCV` (`n_iter=20`, 5-fold `StratifiedKFold`, `scoring="average_precision"`, `random_state=42`) trên tập train.

*Lưu ý:* Với 21 đặc trưng, cả `"sqrt"` ($\lfloor\sqrt{21}\rfloor = 4$) và `"log2"` ($\lfloor\log_2 21\rfloor = 4$) đều xét 4 đặc trưng tại mỗi nút chia, giúp các cây đa dạng hơn và chạy nhanh hơn `0.5` (10 đặc trưng).

In [4]:
from src.models.random_forest import PARAM_DISTRIBUTIONS, run_tuning

print("Không gian tìm kiếm (PARAM_DISTRIBUTIONS):")
for k, v in PARAM_DISTRIBUTIONS.items():
    print(f"  {k}: {v}")

tuning_result = run_tuning(X_train, y_train, config=config, n_iter=20)
best_params = tuning_result["best_params"]

print(f"\nThời gian tìm kiếm (20 tổ hợp x 5 fold): {tuning_result['search_time_seconds']:.2f}s")
print(f"Best Average Precision (5-fold CV)     : {tuning_result['best_score']:.4f}")
print(f"Tăng so với Baseline                   : +{tuning_result['best_score'] - baseline_cv['average_precision_mean']:.4f}")
print(f"Best params                            : {best_params}")
tuning_result["cv_results_df"].head(10)

Không gian tìm kiếm (PARAM_DISTRIBUTIONS):
  model__n_estimators: [100, 200, 300]
  model__max_depth: [8, 12, 16, 20, None]
  model__min_samples_leaf: [1, 5, 10, 20, 40]
  model__max_features: ['sqrt', 'log2', 0.5]


KeyboardInterrupt: 

## 5. Kết quả mean/std, `best_params`, thời gian và fitted pipeline (T27)

Đánh giá lại bộ `best_params` trên 5-fold CV với đủ 6 chỉ số, huấn luyện bản cuối trên toàn bộ 100% tập train (`183,579` mẫu), xuất `artifacts/random_forest.joblib` và `results/metrics/random_forest.json`, đồng thời vẽ đường cong PR/ROC (out-of-fold) và Top Feature Importances.

In [ ]:
import joblib
from src.models.random_forest import train_and_export

export_result = train_and_export(
    X_train,
    y_train,
    best_params=best_params,
    baseline_ap_mean=baseline_cv["average_precision_mean"],
    config=config,
)
tuned_cv = export_result["validation_cv"]
final_pipeline = export_result["pipeline"]

comparison_df = pd.DataFrame(
    {
        "Metric": metrics_order,
        "Baseline Mean": [baseline_cv[f"{m}_mean"] for m in metrics_order],
        "Tuned Mean": [tuned_cv[f"{m}_mean"] for m in metrics_order],
        "Tuned Std": [tuned_cv[f"{m}_std"] for m in metrics_order],
        "Delta": [tuned_cv[f"{m}_mean"] - baseline_cv[f"{m}_mean"] for m in metrics_order],
    }
)
print(f"Tuned 5-fold CV time : {export_result['cv_time_seconds']:.2f}s")
print(f"Final fit time       : {export_result['final_fit_seconds']:.2f}s")
print(f"Saved pipeline       : {export_result['artifact_path']}")
print(f"Saved metrics JSON   : {export_result['metrics_path']}")
comparison_df

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay
from sklearn.model_selection import cross_val_predict

from src.models.random_forest import build_tuned_pipeline

figures_dir = ROOT / config["paths"]["figures"]
figures_dir.mkdir(parents=True, exist_ok=True)

# 1. Đường cong Precision-Recall và ROC trên out-of-fold (chỉ dùng tập train)
oof_pipeline = build_tuned_pipeline(
    X_train, best_params=best_params, random_state=config["split"]["random_state"], n_jobs=-1
)
oof_proba = cross_val_predict(
    oof_pipeline, X_train, y_train, cv=cv, method="predict_proba", n_jobs=1
)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
PrecisionRecallDisplay.from_predictions(y_train, oof_proba, name="Tuned RF (OOF)", ax=axes[0])
axes[0].axhline(y_train.mean(), linestyle="--", color="gray", label=f"No-skill ({y_train.mean():.3f})")
axes[0].set_title("Precision–Recall Curve (5-fold OOF)")
axes[0].legend()

RocCurveDisplay.from_predictions(y_train, oof_proba, name="Tuned RF (OOF)", ax=axes[1])
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random (0.500)")
axes[1].set_title("ROC Curve (5-fold OOF)")
axes[1].legend()

plt.tight_layout()
pr_roc_path = figures_dir / "random_forest_pr_roc_oof.png"
plt.savefig(pr_roc_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {pr_roc_path}")

# 2. Top 15 Feature Importances (MDI) từ mô hình cuối
importances = pd.Series(
    final_pipeline.named_steps["model"].feature_importances_,
    index=X_train.columns,
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importances.tail(15).plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set_title("Top 15 Feature Importances — Tuned Random Forest (MDI)")
ax.set_xlabel("Mean Decrease in Impurity")
plt.tight_layout()
fi_path = figures_dir / "random_forest_feature_importance.png"
plt.savefig(fi_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved figure: {fi_path}")

In [ ]:
# Kiểm thử nạp lại fitted pipeline từ đĩa (độc lập với biến trong notebook)
loaded_pipeline = joblib.load(export_result["artifact_path"])
sample_proba = loaded_pipeline.predict_proba(X_train.iloc[:5])[:, 1]
print("Reloaded pipeline predict_proba (5 mẫu đầu):", np.round(sample_proba, 4))
print("Metrics JSON đã lưu:")
print(export_result["metrics_path"].read_text(encoding="utf-8"))

## 6. Nhận xét và bàn giao

### 6.1. Nhận xét kết quả thực nghiệm (chỉ trên train/CV, chưa mở test)
- **Hiệu quả tối ưu siêu tham số:** Mô hình mặc định (`max_depth=None`, `min_samples_leaf=1`) bị overfitting mạnh trên 183,579 mẫu, chỉ đạt `AP = 0.3629` và `ROC-AUC = 0.7787`. Sau khi tối ưu (`n_estimators=300`, `max_depth=16`, `min_samples_leaf=20`, `max_features='sqrt'`), **`AP` tăng vọt lên `0.4307 ± 0.0056` (`+0.0678`)**, **`ROC-AUC` tăng lên `0.8125 ± 0.0011` (`+0.0338`)**, và **`Precision(1)` tăng từ `0.4742` lên `0.6122`**.
- **Lưu ý về Recall tại ngưỡng mặc định `0.5`:** `Recall(1)` ở ngưỡng `0.5` giảm nhẹ (`0.1629 → 0.1066`) vì `min_samples_leaf=20` giúp xác suất đầu ra mượt và bảo thủ hơn trên dữ liệu mất cân bằng (`15.29%` lớp 1). Tuy nhiên, `AP` và `ROC-AUC` tăng mạnh chứng tỏ **chất lượng xếp hạng xác suất tốt hơn hẳn**; nếu Random Forest được chọn ở M06, việc hạ ngưỡng quyết định (`decision_threshold`) dựa trên out-of-fold validation sẽ cải thiện mạnh Recall/F1 mà không cần huấn luyện lại.
- **Độ ổn định & thời gian:** Độ lệch chuẩn qua 5 fold rất nhỏ (`AP std = 0.0056`, `ROC-AUC std = 0.0011`); thời gian fit bản cuối trên toàn bộ tập train chỉ `~8 giây`.

### 6.2. Checklist bàn giao cho M05 (Hưng) và M06 (Hoàng)
- [x] Mã nguồn tái lập: `src/models/random_forest.py`
- [x] Notebook đầy đủ 6 nội dung: `notebooks/03_random_forest.ipynb`
- [x] Fitted pipeline (kèm preprocessing M03): `artifacts/random_forest.joblib`
- [x] Kết quả CV & `best_params` đúng schema chung: `results/metrics/random_forest.json`
- [x] Biểu đồ OOF PR/ROC & Feature Importance: `results/figures/random_forest_*.png`
- [x] Xác nhận **không sử dụng tập test** trong toàn bộ quá trình huấn luyện và tối ưu.